# 🎯 Exercise: Gradient Descent From Scratch

**Module 1 · Exercise 2**

Before you let PyTorch and scikit-learn do the work, implement the thing by hand once. This is a rite of passage. It'll take 30–60 minutes and the payoff is that every piece of advice you hear about training models for the rest of your life will make sense.

## What you'll build

1. Synthesize noisy linear data.
2. Define a loss function (mean squared error).
3. Compute its gradient analytically.
4. Run gradient descent to fit a line.
5. Plot the loss curve, the final fit, and the descent trajectory.
6. Experiment with learning rates. Watch what happens when you get it wrong.
7. Extend to multi-feature linear regression.

## Setup

```bash
pip install numpy matplotlib
jupyter lab
```

Then open this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.style.use('dark_background')

## 1. Make some fake data

True relationship: `y = 3x + 2 + noise`. Our model will try to recover the 3 and the 2 without being told.

In [ ]:
n = 100
x = np.random.uniform(-3, 3, size=n)
y_true = 3 * x + 2
y = y_true + np.random.normal(0, 1, size=n)  # add gaussian noise

plt.scatter(x, y, alpha=0.6, label='noisy data')
plt.plot(np.sort(x), 3 * np.sort(x) + 2, 'orange', label='true line (unknown to us)')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.title('Our training data')
plt.show()

## 2. The model and loss

Our model is $\hat{y} = w x + b$. Two parameters: $w$ (slope) and $b$ (intercept).

Mean squared error:

$$L(w, b) = \frac{1}{n} \sum_i (w x_i + b - y_i)^2$$

Why squared? It's differentiable everywhere, penalizes large errors more than small ones, and has a nice gradient.

In [ ]:
def predict(w, b, x):
    return w * x + b

def mse_loss(w, b, x, y):
    preds = predict(w, b, x)
    return np.mean((preds - y) ** 2)

# sanity check: bad guess should have high loss
mse_loss(0, 0, x, y)

## 3. Derive the gradient

We need $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$.

Applying the chain rule (treat the sum as over individual squared errors):

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (w x_i + b - y_i) \cdot x_i$$

$$\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (w x_i + b - y_i)$$

In plain English: the gradient is proportional to the **residuals** (prediction minus target). That's a general pattern in ML — the gradient of a loss is "how wrong you are, pointing in the direction of the mistake."

In [ ]:
def gradient(w, b, x, y):
    preds = predict(w, b, x)
    residuals = preds - y
    n = len(x)
    grad_w = (2 / n) * np.sum(residuals * x)
    grad_b = (2 / n) * np.sum(residuals)
    return grad_w, grad_b

### Quick sanity check: numerical vs analytic gradient

Whenever you derive a gradient by hand, verify it against a finite-difference approximation. This has saved countless ML engineers from shipping broken loss functions.

In [ ]:
def numerical_gradient(w, b, x, y, h=1e-5):
    dw = (mse_loss(w + h, b, x, y) - mse_loss(w - h, b, x, y)) / (2 * h)
    db = (mse_loss(w, b + h, x, y) - mse_loss(w, b - h, x, y)) / (2 * h)
    return dw, db

w_test, b_test = 1.5, 0.3
print('analytic :', gradient(w_test, b_test, x, y))
print('numerical:', numerical_gradient(w_test, b_test, x, y))

If those two lines match to ~5 decimal places, your gradient is right. If not — debug before moving on.

## 4. The descent loop itself

In [ ]:
def gradient_descent(x, y, lr=0.01, n_steps=200, w_init=0.0, b_init=0.0):
    w, b = w_init, b_init
    history = []
    for step in range(n_steps):
        loss = mse_loss(w, b, x, y)
        gw, gb = gradient(w, b, x, y)
        history.append((w, b, loss))
        w -= lr * gw
        b -= lr * gb
    return w, b, history

w_star, b_star, history = gradient_descent(x, y, lr=0.05, n_steps=200)
print(f'Learned w = {w_star:.3f}, b = {b_star:.3f}')
print(f'True    w = 3.000, b = 2.000')

You should have recovered something very close to the true parameters. Not exact, because the data is noisy and we have only 100 samples.

## 5. Plot what happened

In [ ]:
ws, bs, losses = zip(*history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(losses, color='#5b9dff', linewidth=2)
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss'); axes[0].set_title('Loss over training')
axes[0].set_yscale('log')

# Final fit
axes[1].scatter(x, y, alpha=0.5, label='data')
xs = np.linspace(-3, 3, 100)
axes[1].plot(xs, 3 * xs + 2, 'orange', label='true', linewidth=2)
axes[1].plot(xs, w_star * xs + b_star, 'cyan', label='learned', linewidth=2, linestyle='--')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y'); axes[1].legend(); axes[1].set_title('Final fit')

plt.show()

## 6. Visualize the descent trajectory on the loss surface

Because we have only two parameters, we can literally see the loss as a 2D surface and watch the trajectory roll downhill.

In [ ]:
w_range = np.linspace(-1, 5, 80)
b_range = np.linspace(-1, 5, 80)
W, B = np.meshgrid(w_range, b_range)
L = np.array([[mse_loss(wi, bi, x, y) for wi in w_range] for bi in b_range])

plt.figure(figsize=(9, 7))
cs = plt.contourf(W, B, L, levels=30, cmap='viridis')
plt.colorbar(cs, label='loss')
plt.plot(ws, bs, 'o-', color='white', markersize=3, linewidth=1, alpha=0.8, label='trajectory')
plt.plot(3, 2, 'r*', markersize=20, label='true minimum')
plt.xlabel('w (slope)'); plt.ylabel('b (intercept)')
plt.title('Gradient descent rolling down the loss surface')
plt.legend()
plt.show()

Beautiful, right? You can see the descent roll down toward the red star. That red star is the true minimum. The small gap is noise — with infinite data, we'd hit it exactly.

## 7. The learning rate playground

**Your turn.** Try each of the learning rates below. Predict what you'll see before running. Then see if you were right.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, lr in zip(axes, [0.001, 0.05, 0.25]):
    try:
        _, _, hist = gradient_descent(x, y, lr=lr, n_steps=100)
        losses = [h[2] for h in hist]
        ax.plot(losses, linewidth=2)
        ax.set_title(f'lr = {lr}  ·  final loss = {losses[-1]:.3f}')
    except Exception as e:
        ax.set_title(f'lr = {lr}  ·  blew up!')
    ax.set_xlabel('step'); ax.set_ylabel('loss')
plt.suptitle('Learning rate matters a LOT')
plt.tight_layout()
plt.show()

What you should see:
- **lr = 0.001** — converges, but slowly. After 100 steps we're still far from the minimum.
- **lr = 0.05** — converges nicely.
- **lr = 0.25** — diverges. Loss explodes to infinity or NaN.

If you crank the learning rate even higher, you'll see `RuntimeWarning: overflow`. This is what "my loss went to NaN" feels like for real.

### Try it

- Find the largest learning rate that still converges. It'll be around 0.15–0.2 for this problem.
- With a very small `lr=0.0001`, how many steps does it take to converge?

## 8. Sanity check against the closed-form solution

Linear regression has a closed-form solution — no iteration needed. NumPy provides it:

In [ ]:
# Add a column of ones for the intercept
X = np.column_stack([x, np.ones(n)])
params, *_ = np.linalg.lstsq(X, y, rcond=None)
print(f'Closed-form:  w = {params[0]:.4f}, b = {params[1]:.4f}')
print(f'Our descent:  w = {w_star:.4f}, b = {b_star:.4f}')

Should match to 3–4 decimal places. If it doesn't, something is off.

**Why bother with gradient descent if there's a closed form?** Because the closed form exists only for a few special cases (linear regression, ridge regression). Every other model — logistic regression, neural networks, every LLM — is trained by gradient descent (or its cousins). The pattern you just implemented scales to billion-parameter models. The closed form doesn't.

## 9. Challenge: multi-feature linear regression

Extend what you built to $d$ features. The data matrix $X$ is now $(n, d)$, and the weights $w$ are a vector of length $d$.

Model: $\hat{y} = Xw + b$.

Gradient (derive it!):

$$\nabla_w L = \frac{2}{n} X^T (Xw + b - y), \quad \frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

Fill in the function below.

In [ ]:
# Generate multi-feature data: y = [2, -1, 0.5] · x + 1.0 + noise
d = 3
n = 500
X = np.random.randn(n, d)
true_w = np.array([2.0, -1.0, 0.5])
true_b = 1.0
y = X @ true_w + true_b + np.random.normal(0, 0.5, size=n)

def mv_gradient_descent(X, y, lr=0.05, n_steps=500):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    history = []
    for step in range(n_steps):
        preds = X @ w + b           # (n,)
        residuals = preds - y       # (n,)
        loss = np.mean(residuals ** 2)
        # TODO: compute grad_w (shape (d,)) and grad_b (scalar)
        grad_w = ???
        grad_b = ???
        history.append(loss)
        w = w - lr * grad_w
        b = b - lr * grad_b
    return w, b, history

# w_hat, b_hat, hist = mv_gradient_descent(X, y)
# print('learned w:', w_hat)
# print('true    w:', true_w)
# print('learned b:', b_hat, '  true b:', true_b)

### Hints

- `grad_w` should have shape `(d,)` — use `X.T @ residuals / n * 2`.
- `grad_b` is a scalar — `residuals.mean() * 2`.
- Before computing, check `X.shape`, `w.shape`, `residuals.shape` — the shapes must align.
- Compare your result to `np.linalg.lstsq` for the ground truth.

### Stretch

- Add an L2 penalty (ridge regression): $L = \frac{1}{n}\|Xw - y\|^2 + \alpha \|w\|^2$. How does the gradient change?
- Implement mini-batch SGD: at each step, use a random subset of 32 rows instead of all of them.
- Plot the trajectory of `w[0]` and `w[1]` through training (like section 6 but for higher dims — you'll need to pick two components to visualize).

## 10. Reflection

Before you close this notebook, write 2–3 sentences in `../../progress/journal.md` about:

- What clicked?
- What's still fuzzy?
- One thing you want to understand more deeply.

Then update the dashboard and move on.